In [1]:
import cv2
import numpy as np
import joblib
from skimage import feature

In [2]:
svm_lbp = joblib.load("models/svm_lbp_barba.joblib")
scaler_lbp  = joblib.load("models/scaler_lbp_barba.joblib")

WIDTH =  192 # width que usé en el entrenamiento
HEIGHT = 192 # height que usé en el entrenamiento

# Nombres de clases en el orden en que se crearon las carpetas
classlabels = ["con_barba", "sin_barba"]

# Clasificador de caras Viola-Jones de OpenCV (incluido en CV2)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

In [3]:
def lbphist(gray, ncellsx, ncellsy, width, height, lbp_method):
    pxpercellx = int(width / ncellsx)
    pxpercelly = int(height / ncellsy)
    ofx = int((width - int(pxpercellx) * ncellsx) / 2)
    ofy = int((height - int(pxpercelly) * ncellsy) / 2)
    LBPu_hist = []
    for i in range(ncellsy):
        for j in range(ncellsx):
            roi = gray[ofy + i * pxpercelly:ofy + (i + 1) * pxpercelly,
                       ofx + j * pxpercellx:ofx + (j + 1) * pxpercellx]
            lbpimg = feature.local_binary_pattern(roi, 8, 1, method=lbp_method)
            n_bins = 256
            feath, _ = np.histogram(lbpimg, density=False,
                                    bins=n_bins, range=(0, n_bins))
            LBPu_hist = np.concatenate([LBPu_hist, feath])
    return LBPu_hist


def preprocess_lbp_from_gray(gray):
    gray_resized = cv2.resize(gray, (WIDTH, HEIGHT),
                              interpolation=cv2.INTER_AREA)
    feat_lbp = lbphist(gray_resized,
                       ncellsx=3, ncellsy=3,
                       width=WIDTH, height=HEIGHT,
                       lbp_method="nri_uniform")
    desc = feat_lbp.astype("float32").reshape(1, -1)
    # APLICAR scaler exactamente igual que en entrenamiento
    desc_scaled = scaler_lbp.transform(desc)
    return desc_scaled


In [4]:
def infer_single_image(img_path, true_label=None):
    img = cv2.imread(img_path)
    if img is None:
        print("No se pudo leer la imagen:", img_path)
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    desc_scaled = preprocess_lbp_from_gray(gray)

    pred_int = int(svm_lbp.predict(desc_scaled)[0])
    pred_label = classlabels[pred_int]

    print("Imagen:", img_path)
    print("Predicción (int):", pred_int)
    print("Predicción (label):", pred_label)
    if true_label is not None:
        print("Etiqueta real:", true_label)

    vis = img.copy()
    cv2.putText(vis, pred_label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
    cv2.imshow("Inferencia", vis)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


In [ ]:
infer_single_image("dataset_barba/con_barba/10.jpg", true_label="con_barba")

Imagen: dataset_barba/sin_barba/10.jpg
Predicción (int): 1
Predicción (label): sin_barba
Etiqueta real: con_barba


In [5]:
def expand_bbox(x, y, w, h, frame_width, frame_height, factor=1.3):
    cx = x + w / 2.0
    cy = y + h / 2.0
    new_w = w * factor
    new_h = h * factor
    x_new = max(0, int(cx - new_w / 2.0))
    y_new = max(0, int(cy - new_h / 2.0))
    x_new2 = min(frame_width,  int(x_new + new_w))
    y_new2 = min(frame_height, int(y_new + new_h))
    return x_new, y_new, x_new2 - x_new, y_new2 - y_new


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h_frame, w_frame = frame_gray.shape

    faces = face_cascade.detectMultiScale(
        frame_gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        x_exp, y_exp, w_exp, h_exp = expand_bbox(
            x, y, w, h, w_frame, h_frame, factor=1.3)

        face_gray = frame_gray[y_exp:y_exp+h_exp, x_exp:x_exp+w_exp]

        # MISMO PREPROCESADO + SCALER
        desc_scaled = preprocess_lbp_from_gray(face_gray)
        pred = int(svm_lbp.predict(desc_scaled)[0])
        label = classlabels[pred]

        color = (0, 255, 0) if label == "con_barba" else (0, 0, 255)
        cv2.rectangle(frame, (x_exp, y_exp),
                      (x_exp + w_exp, y_exp + h_exp), color, 2)
        cv2.putText(frame, label, (x_exp, y_exp - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imshow("Detector barba / sin barba", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
